# ¿Qué solución conviene para el proyecto de fallas de SSD?

Compara tres soluciones con la misma preparación y la misma división de `pipeline_proyecto.py`:

1. El mejor ensamblaje de la Semana 10: el bosque aleatorio con `min_samples_leaf=2`.
2. Una red neuronal simple (`MLPClassifier`, dos capas ocultas de 32 y 16 neuronas).
3. Un LLM al que se le pasan 30 SSD de entrenamiento con su etiqueta como ejemplos y 10 SSD de prueba sin etiqueta, y que responde una probabilidad de falla para cada uno.

Métrica: average precision (la del proyecto). El LLM solo se mide sobre los 10 SSD del prompt, y los dos modelos también sobre esos 10, para comparar en igualdad de condiciones.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from pipeline_proyecto import COLUMNAS, OBJETIVO, cargar_datos, dividir, preparacion

datos = cargar_datos()
X_ent, X_prueba, y_ent, y_prueba = dividir(datos)
print(f"Entrenamiento: {X_ent.shape}  Prueba: {X_prueba.shape}  tasa de falla {y_ent.mean():.4f}")

Entrenamiento: (141520, 14)  Prueba: (35380, 14)  tasa de falla 0.0594


## Paso 1: el ensamblaje y la red neuronal

In [2]:
resultados = {}
modelos = {}


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    probabilidad = pipeline.predict_proba(X_prueba)[:, 1]
    resultados[nombre] = {
        "solucion": nombre,
        "ap_prueba": round(average_precision_score(y_prueba, probabilidad), 4),
        "auc_prueba": round(roc_auc_score(y_prueba, probabilidad), 4),
        "segundos_entrenamiento": round(segundos, 2),
    }
    modelos[nombre] = pipeline
    print(f"{nombre:28s} AP {resultados[nombre]['ap_prueba']:.4f}   AUC {resultados[nombre]['auc_prueba']:.4f}   {segundos:6.1f} s")


evaluar("bosque min_samples_leaf=2", RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42))
evaluar("red neuronal 32-16", MLPClassifier(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=300, random_state=42))

red = modelos["red neuronal 32-16"].named_steps["modelo"]
parametros_red = sum(c.size for c in red.coefs_) + sum(b.size for b in red.intercepts_)
resultados["red neuronal 32-16"]["parametros"] = int(parametros_red)
print("Parametros de la red:", parametros_red, " capas:", [c.shape for c in red.coefs_])

bosque min_samples_leaf=2    AP 0.5003   AUC 0.8619      2.3 s


red neuronal 32-16           AP 0.3593   AUC 0.8102      7.3 s
Parametros de la red: 1281  capas: [(22, 32), (32, 16), (16, 1)]


## Paso 2: el prompt del LLM

10 SSD de prueba, 5 que fallaron y 5 sanos (al azar, con la semilla 7, mezclados): al azar, con 5,94 % de fallas, saldrían 0 o 1 fallas y la métrica no diría nada. Como ejemplos, 30 SSD de entrenamiento: 15 que fallaron y 15 sanos. El LLM ve solo los contadores crudos que recibe el modelo, no la etiqueta de los 10.

In [3]:
COLUMNAS_PROMPT = ["app", "r_5", "r_183", "r_184", "r_187", "r_195", "r_197", "r_199",
                   "r_program", "r_erase", "n_wearout", "r_12", "r_174"]
prueba = datos.loc[X_prueba.index]
entrenamiento = datos.loc[X_ent.index]

muestra = pd.concat([
    prueba[prueba[OBJETIVO] == 1].sample(5, random_state=7),
    prueba[prueba[OBJETIVO] == 0].sample(5, random_state=7),
]).sample(frac=1, random_state=7)
ejemplos = pd.concat([
    entrenamiento[entrenamiento[OBJETIVO] == 1].sample(15, random_state=7),
    entrenamiento[entrenamiento[OBJETIVO] == 0].sample(15, random_state=7),
]).sample(frac=1, random_state=7)


def fila(ssd):
    return ", ".join(f"{columna}={ssd[columna]}" for columna in COLUMNAS_PROMPT)


prompt = (
    "Cada linea es la lectura SMART de un SSD de un centro de datos (contadores crudos: r_5 sectores "
    "reasignados, r_187 errores no corregibles, r_197 sectores pendientes, r_199 errores CRC, "
    "n_wearout desgaste normalizado, r_12 encendidos; app es la aplicacion del servidor). "
    "En toda la flota falla cerca del 6 % de los discos, pero en las listas de abajo la mitad fallo.\n\n"
    "Ejemplos con su resultado (failure=1 si el disco fallo):\n"
    + "\n".join(f"{fila(ssd)} -> failure={int(ssd[OBJETIVO])}" for _, ssd in ejemplos.iterrows())
    + "\n\nPara cada uno de estos 10 discos, responde solo la probabilidad de que haya fallado, "
    "un numero entre 0 y 1 por linea, en el mismo orden, sin texto adicional:\n"
    + "\n".join(f"{i + 1}. {fila(ssd)}" for i, (_, ssd) in enumerate(muestra.iterrows()))
)
Path("prompt_llm_proyecto.txt").write_text(prompt, encoding="utf-8")
print(f"prompt_llm_proyecto.txt: {len(prompt)} caracteres, unos {len(prompt) / 4:.0f} tokens")

prompt_llm_proyecto.txt: 7031 caracteres, unos 1758 tokens


## Paso 3: las tres soluciones sobre los mismos 10 SSD

`respuestas_llm_proyecto.txt` tiene las 10 probabilidades que respondió el LLM, una por línea. Las respondió Claude Opus 5.5 leyendo solo `prompt_llm_proyecto.txt`, sin ver las etiquetas de esos 10 SSD.

In [4]:
y_muestra = muestra[OBJETIVO]
ruta_respuestas = Path("respuestas_llm_proyecto.txt")
probabilidades = {nombre: modelo.predict_proba(muestra[COLUMNAS])[:, 1] for nombre, modelo in modelos.items()}
if ruta_respuestas.exists():
    probabilidades["LLM (Claude Opus 5.5)"] = np.array(
        [float(linea) for linea in ruta_respuestas.read_text(encoding="utf-8").split() if linea.strip()]
    )
else:
    print("Falta respuestas_llm_proyecto.txt: pasa prompt_llm_proyecto.txt a un LLM y guarda sus 10 numeros")

comparacion = pd.DataFrame({"failure": y_muestra.to_numpy(), **{n: p.round(3) for n, p in probabilidades.items()}})
print(comparacion.to_string(index=False))
print()
for nombre, probabilidad in probabilidades.items():
    acierto = ((probabilidad >= 0.5).astype(int) == y_muestra.to_numpy()).mean()
    print(f"{nombre:28s} AP en los 10: {average_precision_score(y_muestra, probabilidad):.4f}   aciertos con umbral 0.5: {acierto:.0%}")

 failure  bosque min_samples_leaf=2  red neuronal 32-16  LLM (Claude Opus 5.5)
       0                      0.008               0.022                   0.45
       0                      0.114               0.196                   0.75
       1                      0.781               0.956                   0.95
       1                      0.264               0.084                   0.40
       1                      0.309               0.182                   0.40
       0                      0.006               0.022                   0.40
       0                      0.059               0.037                   0.30
       1                      0.022               0.034                   0.35
       0                      0.007               0.025                   0.40
       1                      0.419               0.145                   0.40



bosque min_samples_leaf=2    AP en los 10: 0.9429   aciertos con umbral 0.5: 60%
red neuronal 32-16           AP en los 10: 0.7862   aciertos con umbral 0.5: 60%
LLM (Claude Opus 5.5)        AP en los 10: 0.6111   aciertos con umbral 0.5: 50%


## Paso 4: el costo de cada solución

Cada día hay que puntuar los 176.900 SSD. Los dos modelos corren en una laptop. El LLM se paga por token: un prompt como este (30 ejemplos y 10 discos) por cada 10 discos, con el precio de entrada de Claude Opus 5.5 (USD 4 por millón de tokens, sin contar la salida).

In [5]:
PRECIO_POR_MILLON_DE_TOKENS = 4.00
DISCOS_POR_DIA = 176_900
tokens_por_prompt = len(prompt) / 4
costo_diario_llm = DISCOS_POR_DIA / 10 * tokens_por_prompt / 1_000_000 * PRECIO_POR_MILLON_DE_TOKENS
print(f"LLM: {tokens_por_prompt:.0f} tokens por prompt, {DISCOS_POR_DIA // 10:,} prompts al dia, "
      f"USD {costo_diario_llm:,.2f} al dia, USD {costo_diario_llm * 30:,.2f} al mes")

inicio = time.perf_counter()
modelos["bosque min_samples_leaf=2"].predict_proba(datos[COLUMNAS])
print(f"Bosque: puntuar los {len(datos):,} SSD tarda {time.perf_counter() - inicio:.1f} s, sin costo por prediccion")

LLM: 1758 tokens por prompt, 17,690 prompts al dia, USD 124.38 al dia, USD 3,731.35 al mes


Bosque: puntuar los 176,900 SSD tarda 0.5 s, sin costo por prediccion


In [6]:
tabla = pd.DataFrame(resultados.values())
tabla["ap_10_ssd"] = [round(average_precision_score(y_muestra, probabilidades[n]), 4) for n in tabla["solucion"]]
if "LLM (Claude Opus 5.5)" in probabilidades:
    tabla.loc[len(tabla)] = {"solucion": "LLM (Claude Opus 5.5)",
                             "ap_10_ssd": round(average_precision_score(y_muestra, probabilidades["LLM (Claude Opus 5.5)"]), 4)}
tabla.to_csv("comparar_proyecto.csv", index=False)
print(tabla.to_string(index=False))

                 solucion  ap_prueba  auc_prueba  segundos_entrenamiento  parametros  ap_10_ssd
bosque min_samples_leaf=2     0.5003      0.8619                    2.32         NaN     0.9429
       red neuronal 32-16     0.3593      0.8102                    7.34      1281.0     0.7862
    LLM (Claude Opus 5.5)        NaN         NaN                     NaN         NaN     0.6111
